# Mamba Paper Study: Single Colab Workflow

This notebook runs the original repository workflow in one linear Colab session:

1. environment and dependency setup
2. Pile-compatible data streaming and GPT-NeoX tokenization
3. Mamba-130M training
4. GPT-2-style Transformer baseline training
5. held-out perplexity and text generation
6. throughput and peak-memory benchmarking
7. plot generation for presentation artifacts

The workflow keeps the original scripts as helper modules and only reduces compute scale for Colab GPUs. The dataset remains The Pile family via `monology/pile-uncopyrighted`.

## 1. Runtime Check

Use a Colab GPU runtime. T4 and L4 are supported for demonstration scale; A100 remains closer to the original project setup.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

try:
    import torch
    print("Python:", sys.version)
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA:", torch.version.cuda)
        print("BF16 supported:", torch.cuda.is_bf16_supported())
except ImportError:
    print("PyTorch is not installed. Colab GPU runtimes normally include CUDA PyTorch.")

PyTorch is not installed. Colab GPU runtimes normally include CUDA PyTorch.


## 2. Get The Repository

If this notebook is opened directly from GitHub in Colab, clone the repo. If you are already inside the repo, this cell leaves the working tree alone.

In [ ]:
REPO_URL = "https://github.com/seriouslysahid/genai-mamba-assignment"
REPO_DIR = Path("/content/genai-mamba-assignment")

if not Path("scripts/train_mamba.py").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    os.chdir(Path.cwd())

print("Working directory:", Path.cwd())
print("Scripts:", sorted(p.name for p in Path("scripts").glob("*.py")))

## 3. Install Dependencies

Recommended Colab strategy:

- keep Colab's preinstalled CUDA PyTorch and Triton to avoid ABI mismatches
- default to `INSTALL_MODE = "pypi"`, which installs `mamba-ssm[causal-conv1d]` with `--no-build-isolation` against the active PyTorch/CUDA stack
- use `INSTALL_MODE = "easywheels"` only if PyPI/source installation is too slow or fails
- uninstall stale Mamba wheels before reinstalling, because a cached or mismatched wheel can link to the wrong CUDA runtime
- run a small Mamba import/model-construction smoke test before training

Official Mamba installation notes require installing PyTorch first and using `--no-build-isolation` so the build sees the CUDA-enabled PyTorch already in the environment.


In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Default: build/install against Colab's active PyTorch/CUDA stack.
# Use "easywheels" only if PyPI/source installation is too slow or fails.
INSTALL_MODE = "pypi"  # "pypi" or "easywheels"
EASYWHEELS_API_KEY = ""  # used only when INSTALL_MODE == "easywheels"
REINSTALL_MAMBA_DEPS = True  # fixes stale/wrong CUDA wheels from earlier attempts


def run_step(label, cmd, check=True):
    print(f"\n[{label}] {' '.join(cmd)}")
    return subprocess.run(cmd, check=check)

install_steps = [
    "project deps",
    "build helpers",
    "mamba deps",
    "import smoke test",
]

with tqdm(total=len(install_steps), desc="dependency setup", unit="step") as pbar:
    run_step("project deps", [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    pbar.update(1)

    run_step("build helpers", [sys.executable, "-m", "pip", "install", "-q", "ninja", "packaging", "wheel"])
    pbar.update(1)

    if REINSTALL_MAMBA_DEPS:
        run_step("remove old mamba deps", [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "mamba-ssm", "causal-conv1d"], check=False)

    if INSTALL_MODE == "easywheels":
        if not EASYWHEELS_API_KEY.strip():
            raise ValueError('Set EASYWHEELS_API_KEY or use INSTALL_MODE = "pypi".')
        cmd = [
            sys.executable, "-m", "pip", "install", "-q",
            "causal-conv1d", "mamba-ssm",
            "--extra-index-url", f"https://{EASYWHEELS_API_KEY.strip()}:@easywheels.io/simple/",
            "--prefer-binary",
        ]
    elif INSTALL_MODE == "pypi":
        cmd = [
            sys.executable, "-m", "pip", "install", "-q",
            "mamba-ssm[causal-conv1d]", "--no-build-isolation",
        ]
    else:
        raise ValueError('INSTALL_MODE must be "pypi" or "easywheels".')

    run_step("mamba deps", cmd)
    pbar.update(1)

    import torch
    import causal_conv1d
    import mamba_ssm
    from mamba_ssm.models.config_mamba import MambaConfig
    from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

    smoke = MambaLMHeadModel(MambaConfig(d_model=64, n_layer=2, vocab_size=128)).to("cuda" if torch.cuda.is_available() else "cpu")
    del smoke
    print("Mamba smoke test passed.")
    pbar.update(1)

subprocess.run([sys.executable, "scripts/check_env.py"], check=False)

## 4. Colab Scale Configuration

The model definitions remain the original Mamba-130M and comparable GPT-2-style Transformer baseline. The defaults below only reduce examples, sequence length, batch size, steps, and benchmark sweep for Colab runtime and VRAM.

In [ ]:
from pathlib import Path
import sys
import torch

sys.path.insert(0, str(Path("scripts").resolve()))

DATASET_NAME = "monology/pile-uncopyrighted"
TOKENIZER_NAME = "EleutherAI/gpt-neox-20b"

# Balanced Colab demo shard.
# Scaled up for T4 Colab Reproduction
# This produces roughly ~200M train tokens
NUM_TRAIN_EXAMPLES = 150_000
NUM_VAL_EXAMPLES = 5_000

# Larger options if the runtime is stable:
#   Colab Pro/L4:      50_000 train, 5_000 val
#   Stretch/A100:     100_000 train, 10_000 val
# Full monology/pile-uncopyrighted is ~335 GB raw / ~176M rows and is not Colab-practical.
SEQ_LEN = 2048
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 32
MAX_STEPS = 2500
EVAL_INTERVAL = 500
EVAL_STEPS = 50
LOG_INTERVAL = 10

def choose_dtype():
    if not torch.cuda.is_available():
        return "float32"
    major, minor = torch.cuda.get_device_capability(0)
    # BF16 kernels require Ampere (sm_80) or newer. T4 is sm_75, so use FP16.
    if major >= 8 and torch.cuda.is_bf16_supported():
        return "bfloat16"
    return "float16"

DTYPE = choose_dtype()
BENCH_SEQ_LENS = [256, 512, 1024, 2048, 4096]
BENCH_BATCH_SIZE = 1

Path("data").mkdir(exist_ok=True)
Path("out").mkdir(exist_ok=True)
print({
    "dataset": DATASET_NAME,
    "num_train_examples": NUM_TRAIN_EXAMPLES,
    "num_val_examples": NUM_VAL_EXAMPLES,
    "seq_len": SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "max_steps": MAX_STEPS,
    "dtype": DTYPE,
    "gpu_capability": torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
})

## 5. Download And Tokenize Data

This preserves the original streaming/tokenization flow: stream The Pile-family text, tokenize with the GPT-NeoX tokenizer, and write `data/train.bin` and `data/val.bin` as uint16 token shards.

In [ ]:
cmd = [
    sys.executable, "scripts/prepare_data.py",
    "--dataset_name", DATASET_NAME,
    "--tokenizer_name", TOKENIZER_NAME,
    "--num_train", str(NUM_TRAIN_EXAMPLES),
    "--num_val", str(NUM_VAL_EXAMPLES),
    "--out_dir", "data",
]
subprocess.run(cmd, check=True)

for path in [Path("data/train.bin"), Path("data/val.bin")]:
    print(path, f"{path.stat().st_size / 1e6:.2f} MB")

## 6. Train Mamba-130M

The helper uses the original Mamba-1-style 24 layer, d_model 768, d_state 16 configuration.

In [ ]:
from types import SimpleNamespace
from train_mamba import train

train(SimpleNamespace(
    model="mamba",
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    seq_len=SEQ_LEN,
    dtype=DTYPE,
    eval_interval=EVAL_INTERVAL,
    eval_steps=EVAL_STEPS,
    log_interval=LOG_INTERVAL,
    warmup_steps=min(20, MAX_STEPS),
    data_dir="data",
    out_dir="out",
))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 7. Train Transformer Baseline

The baseline remains a GPT-2-style Transformer with comparable hidden size and parameter scale.

In [ ]:
train(SimpleNamespace(
    model="transformer",
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    seq_len=SEQ_LEN,
    dtype=DTYPE,
    eval_interval=EVAL_INTERVAL,
    eval_steps=EVAL_STEPS,
    log_interval=LOG_INTERVAL,
    warmup_steps=min(20, MAX_STEPS),
    data_dir="data",
    out_dir="out",
))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 8. Evaluate Checkpoints

Evaluation computes validation negative log likelihood/perplexity and produces text samples from fixed prompts.

In [ ]:
from evaluate_mamba import main as evaluate_main

for model_name in ["mamba", "transformer"]:
    evaluate_main(SimpleNamespace(
        model=model_name,
        ckpt=f"out/{model_name}_final.pt",
        batch_size=BATCH_SIZE,
        max_batches=25,
        data_dir="data",
        out_dir="out",
        seq_len=SEQ_LEN,
        dtype=DTYPE,
    ))
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Benchmark Throughput And Memory

This preserves the original benchmark intent: same random token inputs, same model families, sequence-length sweep, tokens/sec, and peak allocated GPU memory. OOMs are recorded instead of stopping the notebook.

In [ ]:
from benchmark_mamba import run as benchmark_run

benchmark_run(SimpleNamespace(
    seq_lens=BENCH_SEQ_LENS,
    batch_size=BENCH_BATCH_SIZE,
    dtype=DTYPE,
    out_dir="out",
))

## 10. Plot Results

This generates the same presentation-oriented artifacts as the original script: training loss, validation perplexity, throughput, memory, and speedup ratio.

In [ ]:
from plot_results import main as plot_main

plot_main(SimpleNamespace(log_dir="out", out_dir="out"))
print("Output files:")
for path in sorted(Path("out").glob("*")):
    print(" -", path)

## 11. Display Key Outputs

The generated PNGs and JSON files are saved in `out/`. Use these in the academic presentation or download them from the Colab file browser.

In [ ]:
from IPython.display import display, Image

for name in ["training_loss.png", "val_perplexity.png", "throughput.png", "memory.png", "speedup.png"]:
    path = Path("out") / name
    if path.exists():
        print(name)
        display(Image(filename=str(path)))